In [ ]:
# ============================================
# IDENTIFIABILITY ANALYSIS PIPELINE
# Full statistical workflow
# ============================================

In [ ]:

# 1.Setup and Imports
import numpy as np
import pandas as pd
import scipy.io as sio

# Stats
from scipy.stats import ttest_rel
import statsmodels.api as sm
from statsmodels.stats.anova import AnovaRM
import statsmodels.formula.api as smf

# Bayesian
import pymc as pm
import arviz as az

# Plotting (optional)
import matplotlib.pyplot as plt

print("Libraries loaded.")

In [ ]:
# 2.Load paths
data_dir = "/path/to/Identifiability_results/"

conditions = ["anodal", "cathodal", "sham"]
days = ["day1", "day5"]

all_rows = []

for cond in conditions:
    for day in days:

        # Expected file naming convention
        file_path = f"{data_dir}/Identifiability_{cond}_{day}.mat"

        print(f"Loading: {file_path}")

        mat = sio.loadmat(file_path)

        # Extract variables
        Iself = mat["Iself"].flatten()
        Iothers = mat["Iothers_subject"].flatten()

        # Loop over subjects
        for i in range(len(Iself)):
            all_rows.append({
                "subject": i,                # ⚠️ assumes consistent ordering
                "condition": cond,
                "day": day,
                "Iself": Iself[i],
                "Iothers": Iothers[i],
                "Idiff": Iself[i] - Iothers[i]
            })

# Convert to DataFrame
df = pd.DataFrame(all_rows)

# Convert to categorical
df["condition"] = df["condition"].astype("category")
df["day"] = df["day"].astype("category")

print("\nData preview:")
print(df.head())

In [ ]:
# 3.Sanity Checks
print("\nSubjects per condition/day:")
print(df.groupby(["condition","day"])["subject"].nunique())

print("\nMean Idiff per condition:")
print(df.groupby("condition")["Idiff"].mean())

print("\nMean Idiff per condition x day:")
print(df.groupby(["condition","day"])["Idiff"].mean())

In [ ]:
# 4.Running Paired T-Tests
print("\n=== PAIRED T-TESTS (Iself vs Iothers) ===")

for cond in conditions:
    sub = df[df["condition"] == cond]

    t, p = ttest_rel(sub["Iself"], sub["Iothers"])

    print(f"{cond}: t = {t:.3f}, p = {p:.5f}")

In [ ]:
# 5.Running Repeared Measures ANOVA
print("\n=== REPEATED MEASURES ANOVA ===")

anova = AnovaRM(
    df,
    depvar="Idiff",
    subject="subject",
    within=["condition", "day"]
)

anova_results = anova.fit()

print(anova_results)

In [ ]:
# 6.Linear Mixed Model
print("\n=== LINEAR MIXED MODEL ===")

# Model: Idiff ~ condition * day + (1|subject)
model = smf.mixedlm("Idiff ~ condition * day", df, groups=df["subject"])
lme = model.fit()

print(lme.summary())

In [ ]:
# 7.Prepping for Bayesian Staistical Analyses
df["cond_idx"] = df["condition"].cat.codes
df["day_idx"] = df["day"].cat.codes
df["subj_idx"] = df["subject"].astype("category").cat.codes

n_cond = df["cond_idx"].nunique()
n_day = df["day_idx"].nunique()
n_subj = df["subj_idx"].nunique()

print(f"Conditions: {n_cond}, Days: {n_day}, Subjects: {n_subj}")

In [ ]:
# 8.Bayesian Hierachical Model
coords = {
    "condition": df["condition"].cat.categories,
    "day": df["day"].cat.categories,
    "subject": np.unique(df["subj_idx"])
}

with pm.Model(coords=coords) as model:

    # Data indices
    cond_idx = pm.Data("cond_idx", df["cond_idx"].values)
    day_idx = pm.Data("day_idx", df["day_idx"].values)
    subj_idx = pm.Data("subj_idx", df["subj_idx"].values)

    # ---- Priors ----
    mu = pm.Normal("mu", 0, 1)

    cond_effect = pm.Normal("cond_effect", 0, 0.5, dims="condition")
    day_effect = pm.Normal("day_effect", 0, 0.5, dims="day")

    interaction = pm.Normal("interaction", 0, 0.5, dims=("condition", "day"))

    sigma = pm.HalfNormal("sigma", 1)

    sigma_subj = pm.HalfNormal("sigma_subj", 1)
    subj_offset = pm.Normal("subj_offset", 0, sigma_subj, dims="subject")

    # ---- Linear model ----
    mu_obs = (
        mu
        + cond_effect[cond_idx]
        + day_effect[day_idx]
        + interaction[cond_idx, day_idx]
        + subj_offset[subj_idx]
    )

    # Likelihood
    y = pm.Normal("y", mu=mu_obs, sigma=sigma, observed=df["Idiff"].values)

    print("Sampling... (this may take ~1 min)")
    trace = pm.sample(2000, tune=2000, target_accept=0.9)

In [ ]:
# 9.Posterior Summary
summary = az.summary(
    trace,
    var_names=["cond_effect", "day_effect", "interaction"],
    hdi_prob=0.95
)

print(summary)

In [ ]:
# 10.Probability Statements
posterior = trace.posterior

# Flatten chains
cond_samples = posterior["cond_effect"].values.reshape(-1, n_cond)
day_samples = posterior["day_effect"].values.reshape(-1, n_day)
interaction_samples = posterior["interaction"].values.reshape(-1, n_cond, n_day)

cond_labels = df["condition"].cat.categories
day_labels = df["day"].cat.categories

def prob_greater(a, b):
    return np.mean(a > b)

# ---- CONDITION COMPARISONS ----
print("\n=== CONDITION COMPARISONS ===")
for i in range(n_cond):
    for j in range(i+1, n_cond):
        p = prob_greater(cond_samples[:, i], cond_samples[:, j])
        print(f"P({cond_labels[i]} > {cond_labels[j]}) = {p:.3f}")

# ---- DAY EFFECT ----
print("\n=== DAY EFFECT ===")
p_day = prob_greater(day_samples[:, 1], day_samples[:, 0])
print(f"P(Day5 > Day1) = {p_day:.3f}")

# ---- INTERACTION ----
print("\n=== CONDITION-SPECIFIC DAY EFFECTS ===")

for c in range(n_cond):
    diff = interaction_samples[:, c, 1] - interaction_samples[:, c, 0]
    p = np.mean(diff > 0)

    print(f"P(Day5 > Day1 | {cond_labels[c]}) = {p:.3f}")